In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler    


In [64]:
# Load the datasets
df5 = pd.read_csv('features_5S.csv')
df15 = pd.read_csv('features_15S.csv')
df25 = pd.read_csv('features_25S.csv')

UNDERSTANDING THE DATASET

In [65]:
df5.head()

,date,pump_index,std_rush_order,avg_rush_order,std_trades,std_volume,avg_volume,std_price,avg_price,avg_price_max,hour_sin,hour_cos,minute_sin,minute_cos,symbol,gt
0,2018-12-29 17:00:25,0,-0.002,-0.007,-0.000,0.0,-0.000,-0.001,-0.001,-0.001,-0.998,-0.068,0.000,1.000,BRD,0
1,2018-12-29 17:01:55,0,-0.002,-0.007,-0.001,-0.0,-0.002,-0.001,-0.001,-0.001,-0.998,-0.068,0.106,0.994,BRD,0
2,2018-12-29 17:02:10,0,0.000,0.000,0.000,-0.0,0.000,-0.001,-0.001,-0.001,-0.998,-0.068,0.211,0.977,BRD,0
3,2018-12-29 17:25:00,0,-0.002,-0.007,0.000,0.0,-0.000,-0.001,0.000,0.000,-0.998,-0.068,0.461,-0.887,BRD,0
4,2018-12-29 17:36:05,0,0.000,0.000,-0.000,0.0,-0.000,-0.001,0.000,0.000,-0.998,-0.068,-0.638,-0.770,BRD,0


In [66]:
df15.head()


,date,pump_index,std_rush_order,avg_rush_order,std_trades,std_volume,avg_volume,std_price,avg_price,avg_price_max,hour_sin,hour_cos,minute_sin,minute_cos,symbol,gt
0,2018-12-29 17:00:15,0,0.000,0.000,0.000,0.0,-0.000,-0.001,-0.001,-0.001,-0.998,-0.068,0.000,1.000,BRD,0
1,2018-12-29 17:01:45,0,-0.001,-0.005,-0.001,-0.0,-0.002,-0.001,-0.001,-0.001,-0.998,-0.068,0.106,0.994,BRD,0
2,2018-12-29 17:02:00,0,0.000,0.000,0.000,-0.0,-0.003,-0.001,-0.001,-0.001,-0.998,-0.068,0.211,0.977,BRD,0
3,2018-12-29 17:25:00,0,0.000,0.000,0.000,0.0,-0.000,-0.001,0.000,0.000,-0.998,-0.068,0.461,-0.887,BRD,0
4,2018-12-29 17:36:00,0,0.000,0.000,-0.000,0.0,-0.000,-0.000,0.000,0.000,-0.998,-0.068,-0.638,-0.770,BRD,0


In [67]:
df25.head()

,date,pump_index,std_rush_order,avg_rush_order,std_trades,std_volume,avg_volume,std_price,avg_price,avg_price_max,hour_sin,hour_cos,minute_sin,minute_cos,symbol,gt
0,2018-12-29 17:00:25,0,-0.001,-0.005,0.0,0.000,-0.000,-0.0,-0.001,-0.001,-0.998,-0.068,0.000,1.000,BRD,0
1,2018-12-29 17:01:40,0,0.000,0.000,0.0,0.000,-0.000,-0.0,-0.001,-0.001,-0.998,-0.068,0.106,0.994,BRD,0
2,2018-12-29 17:02:05,0,0.000,0.000,0.0,-0.000,-0.002,-0.0,-0.001,-0.001,-0.998,-0.068,0.211,0.977,BRD,0
3,2018-12-29 17:25:00,0,0.000,0.000,0.0,-0.001,-0.004,-0.0,0.000,0.000,-0.998,-0.068,0.461,-0.887,BRD,0
4,2018-12-29 17:35:50,0,0.000,0.000,-0.0,-0.000,0.000,-0.0,0.000,0.000,-0.998,-0.068,-0.553,-0.833,BRD,0


These datasets consists of high-frequency cryptocurrency market data aggregated in short time windows(5s, 15s, 25s)
Each row represents a summary of trading activity in a short time period for a specific cryptocurrency.

### WHY USE 3 DATASETS(5s, 15s, 25s Windows)
Raw trades were grouped into 5s, 15s and 25s to reduce noise and capture short time behaviour patterns.
### 5-Second window

Captures immediate micro-spikes

Very sensitive to rapid coordinated buying

High noise, but very early signal

### 15-second window
Smoother signal

Balances noise and pattern clarity

Captures short bursts more reliably

### 25s window
Even more stable

Detects sustained abnormal activity

Less noise, but slightly delayed detection

5s window detects initial spikes first, 15 second window captures rapid escalation and 25 second window captures sustained abnormal activity. Basicaly it confirms the trend.

### WHY IS IT SUITABLE FOR DETECTING PUMP AND DUMP SCHEMES?
Sudden buying pressure is captured by std_rush_order

spike in trading activity is captured by std_trades

Volume explosion is captured by std_volume

Rapid price acceleration is captures by std_price, avg_price_max

### COLUMNS
std_rush_order - standardized aggressive order activity.

avg_rush_order - mean aggressive order intensity

std_trades - standardized trade counts

avg_price - mean transactional price.

std_price - abnormal price deviation

avg_price_max - peak price in window

hour_sin, hour_cos, minute_sin, minute_cos preserve cyclical time patterns and prevent discontinuities.

### Target Variable
gt - 0(normal), 1(pump)


In [68]:
df5.value_counts('gt')

gt
0    820990
1       317
dtype: int64

In [69]:
df15.value_counts('gt')

gt
0    583787
1       317
dtype: int64

In [70]:
df25.value_counts('gt')

gt
0    481840
1       317
dtype: int64

All datasets have a class imbalance in the target variable

### 5s-Modelling

In [71]:
X = df5.drop(columns=['gt', 'date', 'symbol'])
y = df5['gt']

### LOGISTIC REGRESSION


Chronological train test split.

The data set is a time series meaning Each row is of a later period than the previous row. A random train test split may lead to training data on the future and testing data on the past. The model would have access to data it shouldn't have. 

A chronological split will help us train on data from the past and test on data from the future.

In [72]:
#chronological split
split_index = int(len(df5) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]
# Create a logistic regression pipeline
logistic_pipeline = Pipeline([
    ('model', LogisticRegression(
        class_weight = 'balanced', #handles class imbalance
        max_iter = 1000,
        random_state = 42
        ))
])
#fit the model
logistic_pipeline.fit(X_train, y_train)
#make predictions
y_pred = logistic_pipeline.predict(X_test)
#evaluate the model
print("\nClassification Report:\n",classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    164202
           1       0.16      0.97      0.28        60

    accuracy                           1.00    164262
   macro avg       0.58      0.98      0.64    164262
weighted avg       1.00      1.00      1.00    164262



### RANDOM FOREST CLASSIFIER

In [73]:
#pipeline
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(
    n_estimators = 200,
    max_depth = 5,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs = -1
    ))
])
#fit the model
rf_pipeline.fit(X_train, y_train)
#predict
y_pred_rf = rf_pipeline.predict(X_test)
#evaluate the model
print("\nRandom Forest Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    164202
           1       0.28      0.97      0.44        60

    accuracy                           1.00    164262
   macro avg       0.64      0.98      0.72    164262
weighted avg       1.00      1.00      1.00    164262



### XG BOOST

In [74]:
# Calculate imbalance ratio
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 2555.5953307392997


In [75]:
from xgboost import XGBClassifier
#pipeline
xgb_pipeline = Pipeline([
    ('model', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])
#Fit the model
xgb_pipeline.fit(X_train, y_train)
#predict
y_pred_xb = xgb_pipeline.predict(X_test)
#evaluate the model
print("\nXGBoost Classification Report:\n", classification_report(y_test, y_pred_xb))


XGBoost Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    164202
           1       0.77      0.90      0.83        60

    accuracy                           1.00    164262
   macro avg       0.89      0.95      0.92    164262
weighted avg       1.00      1.00      1.00    164262



### 15s Modelling

In [76]:
# defining x and y
X15 = df15.drop(columns=['gt','date','symbol'])
y15 = df15['gt']

### LOGISTIC REGRESSION

In [77]:
#chronological split
split_index15 = int(len(df15)* 0.8)
X15_train, X15_test = X15[:split_index15], X15[split_index15:]
y15_train, y15_test = y15[:split_index15], y15[split_index15:]

#LOGISTIC REGRESSION FOR 15S DATA
#pipeline
logit15_pipeline = Pipeline([
    ('model', LogisticRegression(
        class_weight = 'balanced',
        max_iter = 1000,
        random_state = 42
    ))
])
#fit the model
logit15_pipeline.fit(X15_train, y15_train)
#predict
y_pred15_log = logit15_pipeline.predict(X15_test)
#evaluate
print("\nClassification Report:\n",classification_report(y15_test, y_pred15_log))
    


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    116760
           1       0.34      1.00      0.51        61

    accuracy                           1.00    116821
   macro avg       0.67      1.00      0.75    116821
weighted avg       1.00      1.00      1.00    116821



### RANDOM FOREST 15 SECONDS

In [78]:
#pipeline
rf15_pipeline = Pipeline([
    ('model', RandomForestClassifier(
    n_estimators = 200,
    max_depth = 5,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs = -1
    ))
])
#fit the model
rf15_pipeline.fit(X15_train, y15_train)
#predict
y_pred_rf15 = rf15_pipeline.predict(X15_test)
#evaluate the model
print("\nRandom Forest Classification Report:\n", classification_report(y15_test, y_pred_rf15))


Random Forest Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    116760
           1       0.39      0.97      0.56        61

    accuracy                           1.00    116821
   macro avg       0.70      0.98      0.78    116821
weighted avg       1.00      1.00      1.00    116821



### XG BOOST

In [79]:
# Calculate imbalance ratio
negative15 = (y15_train == 0).sum()
positive15 = (y15_train == 1).sum()

scale_pos_weight15 = negative15 / positive15
print("scale_pos_weight15:", scale_pos_weight15)

scale_pos_weight15: 1824.32421875


In [80]:
#pipeline
xgb15_pipeline = Pipeline([
    ('model', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=scale_pos_weight15,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])
#Fit the model
xgb15_pipeline.fit(X15_train, y15_train)
#predict
y_pred_xb15 = xgb15_pipeline.predict(X15_test)
#evaluate the model
print("\nXGBoost Classification Report:\n", classification_report(y15_test, y_pred_xb15))


XGBoost Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    116760
           1       0.85      0.92      0.88        61

    accuracy                           1.00    116821
   macro avg       0.92      0.96      0.94    116821
weighted avg       1.00      1.00      1.00    116821



### 25 SEC MODELLING

In [81]:
# defining x and y
X25 = df15.drop(columns=['gt','date','symbol'])
y25 = df15['gt']

### LOGISTIC REGRESSION 25 S

In [82]:
#chronological split
split_index25 = int(len(df25)* 0.8)
X25_train, X25_test = X25[:split_index25], X25[split_index25:]
y25_train, y25_test = y25[:split_index25], y25[split_index25:]

#LOGISTIC REGRESSION FOR 15S DATA
#pipeline
logit25_pipeline = Pipeline([
    ('model', LogisticRegression(
        class_weight = 'balanced',
        max_iter = 1000,
        random_state = 42
    ))
])
#fit the model
logit25_pipeline.fit(X25_train, y25_train)
#predict
y_pred25_log = logit25_pipeline.predict(X25_test)
#evaluate
print("\nClassification Report:\n",classification_report(y25_test, y_pred25_log))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    198274
           1       0.35      1.00      0.52       105

    accuracy                           1.00    198379
   macro avg       0.68      1.00      0.76    198379
weighted avg       1.00      1.00      1.00    198379



### RANDOM FOREST CLASSIFIER 25 S

In [83]:
rf25_pipeline = Pipeline([
    ('model', RandomForestClassifier(
    n_estimators = 200,
    max_depth = 5,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs = -1
    ))
])
#fit the model
rf25_pipeline.fit(X25_train, y25_train)
#predict
y_pred_rf25 = rf25_pipeline.predict(X25_test)
#evaluate the model
print("\nRandom Forest Classification Report:\n", classification_report(y25_test, y_pred_rf25))


Random Forest Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    198274
           1       0.41      0.97      0.58       105

    accuracy                           1.00    198379
   macro avg       0.71      0.99      0.79    198379
weighted avg       1.00      1.00      1.00    198379



### XG BOOST

In [84]:
# Calculate imbalance ratio
negative25 = (y25_train == 0).sum()
positive25 = (y25_train == 1).sum()

scale_pos_weight25 = negative25 / positive25
print("scale_pos_weight15:", scale_pos_weight25)

scale_pos_weight15: 1818.4575471698113


In [85]:
#pipeline
xgb25_pipeline = Pipeline([
    ('model', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=scale_pos_weight25,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])
#Fit the model
xgb25_pipeline.fit(X25_train, y25_train)
#predict
y_pred_xb25 = xgb25_pipeline.predict(X25_test)
#evaluate the model
print("\nXGBoost Classification Report:\n", classification_report(y25_test, y_pred_xb25))


XGBoost Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    198274
           1       0.88      0.87      0.87       105

    accuracy                           1.00    198379
   macro avg       0.94      0.93      0.94    198379
weighted avg       1.00      1.00      1.00    198379

